Lets meet JAX – the third big Deep Learning engine, next to TensorFlow and PyTorch.

JAX is **NumPy that runs on CPU, GPU and TPU**, plus a few *transformations* you can apply to your own Python functions:

1. `jax.grad` – automatic gradients
2. `jax.jit` – compilation with XLA for speed
3. `jax.vmap` – automatic batching

At the end we'll train the same Keras model as in Class 1 – this time with JAX underneath.

More: [JAX documentation](https://docs.jax.dev/) and the [JAX tutorials](https://docs.jax.dev/en/latest/tutorials.html).

## Setup

On Windows and macOS `pip install jax` gives you the CPU version – that's enough for this notebook. GPU support needs Linux or WSL (`pip install -U "jax[cuda12]"`). **Google Colab has JAX preinstalled** and lets you pick a GPU or TPU runtime.

If you installed everything from the repository's `requirements.txt`, JAX is already there. On some Windows machines *Smart App Control* blocks JAX's DLL and `import jax` fails with `DLL load failed ... blocked`. In that case simply run this notebook in Google Colab.

In [ ]:
%pip install -q jax "keras>=3"

We tell Keras to use JAX **before** importing it for the first time. If you already imported Keras in this kernel, restart the kernel.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import time
import numpy as np
import jax
import jax.numpy as jnp

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

## 1. `jax.numpy` – NumPy on accelerators

Almost everything you know from NumPy works the same way, just with `jnp` instead of `np`.

In [ ]:
x = jnp.arange(10.0)
print(x * 2)
print(jnp.dot(x, x))
print(jnp.mean(x), jnp.std(x))

**Difference #1: JAX arrays are immutable.** You can't change an element in place. Instead you create a new array with `.at[...].set(...)`.

Why? Functions that don't modify their inputs ("pure functions") are much easier to compile and to split across many devices.

In [ ]:
x_np = np.arange(5.0)
x_np[0] = 100.0
print("NumPy:", x_np)

x = jnp.arange(5.0)
try:
    x[0] = 100.0
except TypeError as e:
    print("JAX:", e)

x = x.at[0].set(100.0)
print("JAX with .at[]:", x)

**Difference #2: randomness is explicit.** There is no hidden global random state. You create a *key* and split it every time you need new random numbers.

Remember the experiments notebook from Class 2? In JAX reproducibility is built in: the same key always gives the same numbers.

In [ ]:
key = jax.random.key(42)
key, subkey = jax.random.split(key)
print(jax.random.normal(subkey, (3,)))

# The same key always gives the same numbers
print(jax.random.normal(subkey, (3,)))

## 2. `jax.grad` – gradients of any Python function

In Class 1 we used `tf.GradientTape` to get the derivative of `y = 2x + 3`. In JAX, `jax.grad(f)` simply returns **a new function** that computes the gradient of `f`.

In [ ]:
def f(x):
    return 2 * x + 3

df = jax.grad(f)
print(df(0.0))   # dy/dx = 2

In [ ]:
def g(x):
    return x ** 3

print(jax.grad(g)(2.0))             # 3x^2 = 12
print(jax.grad(jax.grad(g))(2.0))   # 6x = 12 – gradients of gradients, for free

Now something closer to Deep Learning: a linear regression model `y = x·w + b` with an MSE loss (Class 2!).

In JAX, the model parameters are just a Python dictionary (JAX calls such nested structures *pytrees*), and `jax.grad` returns gradients with exactly the same structure.

In [ ]:
key, k1, k2 = jax.random.split(key, 3)
X = jax.random.normal(k1, (200, 3))
true_w = jnp.array([1.5, -2.0, 0.5])
y = X @ true_w + 1.0 + 0.1 * jax.random.normal(k2, (200,))

def predict(params, x):
    return x @ params["w"] + params["b"]

def mse(params, x, y):
    return jnp.mean((predict(params, x) - y) ** 2)

params = {"w": jnp.zeros(3), "b": 0.0}
grads = jax.grad(mse)(params, X, y)
grads

And this is gradient descent written by hand – the same loop Keras runs for you inside `fit()`:

1. compute the gradients of the loss,
2. move every parameter a small step against its gradient.

In [ ]:
learning_rate = 0.1

for step in range(100):
    grads = jax.grad(mse)(params, X, y)
    params = jax.tree_util.tree_map(lambda p, g: p - learning_rate * g, params, grads)
    if step % 20 == 0:
        print(f"step {step:3d}  loss {mse(params, X, y):.4f}")

print("learned w:", params["w"], " b:", params["b"])
print("true    w:", true_w, " b: 1.0")

## 3. `jax.jit` – compile for speed

Normally Python runs your function line by line, and each operation is sent to the device separately. `jax.jit` traces the function once, hands it to the **XLA compiler**, and turns it into one fused, optimized program.

Two things to know when timing JAX:
1. The **first call is slow** – that's the compilation.
2. JAX runs **asynchronously** – it returns before the computation is done. `.block_until_ready()` waits for the result, so we measure the real time.

In [ ]:
def deep_stack(x, w):
    for _ in range(20):
        x = jnp.tanh(x @ w)
    return x

deep_stack_jit = jax.jit(deep_stack)

key, k1, k2 = jax.random.split(key, 3)
x = jax.random.normal(k1, (1000, 1000))
w = jax.random.normal(k2, (1000, 1000)) / 30

def timeit(fn, *args, repeats=5):
    fn(*args).block_until_ready()          # warm-up (and compilation for the jitted version)
    start = time.perf_counter()
    for _ in range(repeats):
        fn(*args).block_until_ready()
    return (time.perf_counter() - start) / repeats

print(f"without jit: {timeit(deep_stack, x, w) * 1000:.1f} ms")
print(f"with jit:    {timeit(deep_stack_jit, x, w) * 1000:.1f} ms")

How big the speed-up is depends on your hardware – on a GPU or TPU it's usually much larger than on a laptop CPU.

**The catch:** `jit` traces your function with placeholder values (*tracers*), not real numbers. Python code inside it – like `print` – runs only once, during tracing. And a Python `if` on the *value* of an array won't work, because during tracing there is no value yet.

In [ ]:
@jax.jit
def scaled(x):
    print("Tracing! x is:", x)   # runs only during compilation
    return x * 2

print(scaled(jnp.ones(3)))
print(scaled(jnp.ones(3)))       # no "Tracing!" this time – the compiled version is reused

## 4. `jax.vmap` – automatic batching

Often it's easiest to write a function for **one** example. `jax.vmap` turns it into a function for a whole batch – without writing a loop, and as fast as if you had written the batched version yourself.

In [ ]:
def predict_one(w, x):          # x is ONE sample with 3 features
    return jnp.dot(w, x)

w = jnp.array([1.0, 2.0, 3.0])
batch = jax.random.normal(k1, (128, 3))

# in_axes: don't batch over w (None), batch over the first axis of x (0)
predict_batch = jax.vmap(predict_one, in_axes=(None, 0))
print(predict_batch(w, batch).shape)

Transformations can be combined. Here is something that is surprisingly hard in most frameworks: **the gradient for every single example separately** (useful e.g. for finding mislabeled or the most "influential" samples).

`jax.grad(mse)` computes the gradient for one example, `jax.vmap` runs it for all 200 at once.

In [ ]:
per_example_grads = jax.vmap(jax.grad(mse), in_axes=(None, 0, 0))(params, X, y)
print(per_example_grads["w"].shape)   # (200, 3): one gradient per example
print(per_example_grads["b"].shape)   # (200,)

## 5. Keras on the JAX backend

You don't have to write training loops by hand to use JAX. Keras 3 can run on JAX, TensorFlow or PyTorch – we set `KERAS_BACKEND="jax"` at the top of this notebook.

This is **the model from Class 1** (`keras-intro.ipynb`). The model code is the same – only the engine underneath changed. With the JAX backend Keras compiles the training step with `jax.jit` for you.

In [ ]:
import keras
print("Keras backend:", keras.backend.backend())

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape((60000, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((10000, 28 * 28)).astype("float32") / 255

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28 * 28,)),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

start = time.perf_counter()
model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1)
print(f"Training took {time.perf_counter() - start:.1f} s")

test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"test_acc: {test_acc:.4f}")

## 6. When would you use JAX?

1. **Keras on JAX** – the easiest way in. Same code, often faster training, and access to TPUs.
2. **Pure JAX** – when you need full control: research on new architectures or optimizers, physics simulations, anything that's "just math" and needs gradients.
3. **The JAX ecosystem** – for bigger projects people use [Flax](https://flax.readthedocs.io/) (neural network layers), [Optax](https://optax.readthedocs.io/) (optimizers) and [Orbax](https://orbax.readthedocs.io/) (checkpoints).

| You want to... | TensorFlow | PyTorch | JAX |
|---|---|---|---|
| Get gradients | `tf.GradientTape` | `loss.backward()` | `jax.grad(f)` |
| Make it fast | `tf.function` | `torch.compile` | `jax.jit` |
| Batch a per-example function | `tf.vectorized_map` | `torch.vmap` | `jax.vmap` |

## Your turn

1. Change the size of `x` and `w` in the `jit` example (100, 2000, 4000). When is `jit` worth it the most?
2. Run this notebook in Colab with a GPU, then a TPU runtime. Compare the `jit` times.
3. In the linear regression, try learning rates 0.01, 0.1 and 1.0. What happens with 1.0?
4. Restart the kernel, set `KERAS_BACKEND` to `"tensorflow"` and run section 5 again. Then `"torch"`. Is the accuracy the same? The training time?